# 05 · Optimise the interface geometry

## Objective

The objective is to determine the minimum energy configuration of the interface by performing geometry optimisation. This includes comparing the energies of interfaces in which slabs are translated relative to each other in x, y, z direction. 

Here, two starting offsets in x, y are using for a LiNbO3/SiO2 interface. Each candidate is relaxed with the LiNbOSi-finetuned model to select the lowest-energy successful result.

This notebook prints the candidate parameters and final energies. It saves the selected interface for Notebook 06. The demonstration settings do not ensure converged geometries.

## Prerequisites
Before execution, make sure these dependencies are met,

1. Install the notebook and MLIP dependencies:

   ```bash
   uv sync --group notebooks --group mlip
   ```

2. Set `[programs].mlip_models_dir` in `crystal.toml` to the `mlip_models` directory. The `LiNbOSi-finetuned` files must be available.

The default input is the interface generated with Notebook 04. You can start this notebook independently. It uses the bundled model, not the model trained in Notebook 03.

## 1. Import the optimisation tools

This step imports the existing geometry-optimisation maker, settings, and result type. It imports the storage tools for local execution and ASE for the interface files.

In [ ]:
from ase.io import read, write
from jobflow.core.store import JobStore
from jobflow.managers.local import run_locally
from maggma.stores import MemoryStore
from monty.json import jsanitize
from monty.serialization import dumpfn
from pymatgen.io.ase import AseAtomsAdaptor

from crystal import CRYSTAL_ROOT
from crystal.flows.interface_geometry_optimisation_maker import (
    InterfaceGeometryOptimisationMaker,
)
from crystal.schemas.interface_geometry_optimisation import (
    GeometryOptimisationResult,
    InterfaceGeometryOptimisationInputs,
    InterfaceGeometryOptimisationSimulationSettings,
)
from crystal.schemas.mlip import MlipRelaxMakerSettings, MlipRelaxSimulationSettings
from crystal.schemas.mlip_model_ref import MlipModelRef, QetModelRegistry
from crystal.structure.interfaces.builder import Interface
from crystal.structure.interfaces.settings import InterfaceAssemblySettings


## 2. Set the input path and relaxation parameters

This step selects the input interface and bundled model. Set `INTERFACE_PATH` to `DATA_DIR / "interface.extxyz"` to use the file generated in Notebook 04 instead of the bundled input.

This step sets two fractional in-plane offsets, `(0.0, 0.0)` and `(0.5, 0.0)`, with one gap multiplier and 15 Å vacuum. It sets a maximum of 100 relaxation steps and a force threshold of 0.1 eV/Å. The step limit can stop relaxation before the force threshold is reached. Increase the step limit and check convergence before scientific use.

The local run creates a separate directory for each job. The selected-interface file and result JSON file are overwritten after a successful run. No run flag is used.

In [ ]:
DATA_DIR = CRYSTAL_ROOT / "examples/results"
INTERFACE_PATH = CRYSTAL_ROOT / "examples/data/mock_interface.extxyz"
RUN_DIRECTORY = DATA_DIR / "geometry-runs"
SELECTED_INTERFACE_PATH = DATA_DIR / "selected_interface.extxyz"
GEOMETRY_RESULT_PATH = DATA_DIR / "geometry.json"

MAX_RELAX_STEPS = 100
FORCE_THRESHOLD_EV_PER_ANGSTROM = 0.1
OFFSETS = [(0.0, 0.0), (0.5, 0.0)]
GAP_MULTIPLIERS = [1.0]
INTERFACE_VACUUM_ANGSTROM = 15.0

model = MlipModelRef(registry_model=QetModelRegistry.finetuned_linbosi)


## 3. Read the starting interface

This step reads the extxyz file and reconstructs the `Interface` object from its metadata. The bundled input is a 128-atom LiNbO3/SiO2 starting geometry. It has not been relaxed or scientifically validated.

This step prints the input path, atom count, and model path. It does not run a relaxation.

In [ ]:
if not INTERFACE_PATH.is_file():
    msg = f"No interface file at {INTERFACE_PATH}. Check the path or complete Notebook 04."
    raise FileNotFoundError(msg)
ase_atoms = read(INTERFACE_PATH.as_posix(), format="extxyz")
ase_atoms.info = jsanitize(ase_atoms.info, strict=True)
interface = Interface.from_structure(AseAtomsAdaptor.get_structure(ase_atoms))
if interface.interface_properties.get("synthetic", False):
    print(interface.interface_properties["notice"])
print("Input interface:", INTERFACE_PATH)
print("Interface atoms:", len(interface))
print("MLIP model:", model.model_path)


## 4. Build the interface candidates and workflow

This step uses `InterfaceGeometryOptimisationInputs` to assemble the offset candidates from the matched slabs. It records each candidate's zero-based index, starting fractional offset, and starting gap in Å. These values describe the actual candidates, not parameters reconstructed after the run.

This step builds the workflow with `InterfaceGeometryOptimisationMaker`. The existing workflow relaxes each candidate and selects the lowest-energy successful result. It does not select an unrelaxed fallback if all candidates fail.

In [ ]:
inputs = InterfaceGeometryOptimisationInputs(
    interface=interface,
    assembly_settings=InterfaceAssemblySettings(
        offsets=OFFSETS,
        gap_multipliers=GAP_MULTIPLIERS,
        vacuum=INTERFACE_VACUUM_ANGSTROM,
    ),
)
candidate_parameters = [
    {
        "candidate_index": index,
        "offset": candidate.in_plane_offset.tolist(),
        "gap_angstrom": candidate.gap,
    }
    for index, candidate in enumerate(inputs.interface_candidates)
]
flow = InterfaceGeometryOptimisationMaker(
    simulation_settings=InterfaceGeometryOptimisationSimulationSettings(
        relax_maker_settings=MlipRelaxMakerSettings(
            simulation_settings=MlipRelaxSimulationSettings(
                model=model,
                max_steps=MAX_RELAX_STEPS,
                force_threshold_in_ev_per_angstrom=FORCE_THRESHOLD_EV_PER_ANGSTROM,
            ),
        ),
    ),
).make(inputs)
print("Interface candidates:", len(candidate_parameters))


## 5. Run the geometry optimisation locally

This step creates an in-memory JobStore and runs the workflow on this computer. Each job writes its calculation files to a separate directory under `results/geometry-runs/`. Wait for the cell to finish before you retrieve the result.

The results remain available through `store` while this kernel is active. A kernel restart removes the in-memory records. The calculation files remain on disk.

In [ ]:
RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
store = JobStore(MemoryStore())
run_locally(
    flow,
    store=store,
    root_dir=RUN_DIRECTORY,
    create_folders=True,
    ensure_success=True,
)


## 6. Retrieve and save the selected interface

This step resolves the workflow output from the in-memory store. It reconstructs the selected interface and preserves the metadata and film/substrate labels in `results/selected_interface.extxyz`. Notebook 06 can use this file as its input.

This step also saves `results/geometry.json` with the model reference, candidate parameters, energies, and selected result. This JSON file is a result record, not an interface input file. Both output files are overwritten each time this cell runs.

In [ ]:
resolved_result = flow.output.resolve(store)
if resolved_result is None:
    msg = f"No geometry result is available. Inspect the candidate jobs under {RUN_DIRECTORY}."
    raise RuntimeError(msg)
geometry = GeometryOptimisationResult.model_validate(resolved_result)
selected_interface = Interface.from_structure(geometry.selected_structure)
selected_atoms = AseAtomsAdaptor.get_atoms(selected_interface)
selected_atoms.info = jsanitize(selected_atoms.info, strict=True)
write(SELECTED_INTERFACE_PATH.as_posix(), selected_atoms, format="extxyz")
dumpfn(
    {
        "model": model,
        "geometry": geometry,
        "candidate_parameters": candidate_parameters,
    },
    GEOMETRY_RESULT_PATH,
)
print("Saved selected interface:", SELECTED_INTERFACE_PATH)
print("Saved geometry result:", GEOMETRY_RESULT_PATH)


## 7. Compare the candidate energies

This step prints each candidate's index, starting offset, starting gap in Å, and final total energy in eV. It includes failed candidates and their error messages. The final line identifies the selected interface index.

This step uses the candidate parameters recorded before execution. You can repeat it without running the relaxations again. A successful result record does not by itself confirm that the force threshold was reached; inspect the relaxation logs before scientific use.

In [ ]:
parameters_by_index = {
    item["candidate_index"]: item for item in candidate_parameters
}
print("Index | starting offset (fractional) | starting gap (Å) | energy (eV)")
for record in geometry.candidate_records:
    parameters = parameters_by_index[record.candidate_index]
    offset = parameters["offset"]
    gap_angstrom = parameters["gap_angstrom"]
    energy = (
        f"{record.final_relax_energy:.6f}"
        if record.final_relax_energy is not None
        else "failed"
    )
    print(f"{record.candidate_index} | {offset} | {gap_angstrom:.6f} | {energy}")
    if record.error_message:
        print(f"  Error: {record.error_message}")
print(f"Selected interface index (0-based): {geometry.selected_candidate_index}")


## Next step: calculate MD work of adhesion

This step prepares the input for [Notebook 06](06_mlip_work_of_adhesion.ipynb). Set that notebook's `INTERFACE_PATH` to `DATA_DIR / "selected_interface.extxyz"` to use the selected geometry with the same bundled model. Keep the interface metadata in the file. Geometry optimisation does not establish thermal equilibration.